# Cost per asset: Santa's cards

Run this notebook on your **laptop**. No GPU and no credits needed.

It reads batch runs that made Santa's animated gift cards for 20 kids, on four GPUs. It turns the run times into a cost per card, first at **regular** prices, then at **spot** prices, and compares the most expensive step with a per-call API for the same model.

You're done when you have:

- the per-step table for one run: model load time and median time per card;
- the cost per card at N = 1, 10, 20 and 200 at regular prices, extrapolated from that one run;
- the cost per card on each GPU, and the clip breakeven against the API, from our example runs;
- what spot prices change.

You don't run anything new: one run is enough (ours, or your own 10-kid batch Job), and the other GPUs are our runs. Section 5 is an optional practice with your own run.

Guide: [05-measuring-costs.md](../05-measuring-costs.md).

In [ ]:
import json
import statistics
import sys
import urllib.request
from pathlib import Path

MODULE = "01-consistent-media-generation-on-serverless-gpus"
candidates = [Path.cwd(), Path.cwd() / "code", Path.cwd() / MODULE / "code"]
found = [p for p in candidates if (p / "common.py").is_file()]
assert found, "Open this notebook from the repository (its code/ folder)"
CODE = found[0].resolve()
RUNS_DIR = CODE.parent / "assets" / "example_runs"
sys.path.insert(0, str(CODE))

## 0. Prices: update this cell first

Every cost in this notebook comes from this one cell. Prices change: when they do, update the cell and run the notebook again.

- **Regular:** the on-demand price from [Compute pricing](https://docs.nebius.com/compute/resources/pricing). H100, H200 and RTX PRO 6000 list one price per GPU hour. L40S lists GPU, vCPU and RAM separately: for `1gpu-8vcpu-32gb`, add GPU + 8 × vCPU + 32 × RAM.
- **Spot:** the price for preemptible GPUs, which our example runs used. It can change as often as every 15 minutes, up to one cent below the regular price. Read it from the [spot-price API](https://docs.nebius.com/signup-billing/spot-prices) (the next cell prints it). L40S isn't in the spot API: its preemptible price is fixed on the pricing page.
- **API:** one per-call price for the same model as our clip step, Wan 2.2 A14B image-to-video, from fal's model page. We compare the clip step only.

`metrics_batch.json` starts when the worker starts. The time before it, from **Create** until the container runs, is the startup. It isn't in the file, so we type it here: one H100 Job measured it on 2026-11-14.

In [ ]:
PRICE_DATE = "2026-11-15"  # when the prices below were read
SPOT_SNAPSHOT = "2026-11-14T15:05Z"  # snapshotTime the spot-price API returned that day

# USD per hour for one GPU and the preset the run used
PRICES = {
    "H100": {"file": "h100.json", "platform": "gpu-h100-sxm", "region": "eu-north1",
             "regular": 4.50, "spot": 2.07},
    "H200": {"file": "h200.json", "platform": "gpu-h200-sxm", "region": "eu-north1",
             "regular": 5.40, "spot": 2.48},
    # RTX PRO 6000 (gpu-rtx6000-a): our run was in uk-south2
    "RTX PRO 6000": {"file": "rtx-pro-6000.json", "platform": "gpu-rtx6000-a",
                     "region": "uk-south2", "regular": 1.80, "spot": 1.38},
    # L40S: no spot price; regular and preemptible are fixed, GPU + 8 vCPU + 32 GiB RAM
    "L40S": {"file": "l40s.json", "platform": "gpu-l40s-a", "region": "eu-north1",
             "regular": 1.35 + 8 * 0.012 + 32 * 0.0032, "spot": 0.65 + 8 * 0.006 + 32 * 0.0016},
}

# Per-call API for the clip step: the same model, Wan 2.2 A14B image-to-video
API = {
    "provider": "fal",
    "model": "fal-ai/wan/v2.2-a14b/image-to-video",
    "link": "https://fal.ai/models/fal-ai/wan/v2.2-a14b/image-to-video",
    "resolution": "480p",
    "usd_per_video_s": 0.04,  # 480p; the page lists 0.06 for 580p and 0.08 for 720p
    "frames_per_video_s": 16,  # fal counts video seconds at 16 frames per second
    "checked": "2026-11-15",
}

T_START_S = 157.7  # s, Create -> container running; one H100 Job, 2026-11-14
MAIN_GPU = "H100"  # the run sections 1-3 use

In [ ]:
# Optional: today's spot prices for the platforms above, to update PRICES
url = "https://api.nebius.cloud/billing/v1alpha1/spot-price"
with urllib.request.urlopen(url, timeout=30) as response:
    spot = json.load(response)
platforms = {p["platform"] for p in PRICES.values()}
print("snapshot:", spot["snapshotTime"])
for item in spot["items"]:
    if item["computeInstance"]["platform"] in platforms:
        price = item["price"]
        print(f"{item['computeInstance']['platform']:<15}{item['region']:<13}"
              f"{price['value']} {price['currency']} per {price['unitTitle']}, "
              f"since {item['effectiveTime']}")

## 1. Load one run

`metrics_batch.json` is written by the worker inside the Job. For each step it has the model load time (`load_s`) and the time for each kid (`latency_s`).

To use your own run, for example the 10-kid batch Job: download its `metrics_batch.json` from your bucket, put it in `assets/example_runs/`, add a row to `PRICES` and set `MAIN_GPU` to it. One run gives `F` and `c`; the sections below extrapolate to any N.

In [ ]:
def load_run(gpu: str) -> dict:
    """Read one run's metrics_batch.json from assets/example_runs/."""
    return json.loads((RUNS_DIR / PRICES[gpu]["file"]).read_text())


m = load_run(MAIN_GPU)
steps = m["steps"]
kids = list(next(iter(steps.values()))["latency_s"])
print(f"{m['n']} kids on {m['gpu']}, steps: {', '.join(steps)}")
print(f"worker runtime: {m['runtime_s']:.1f} s, started {m['started_at']}")

## 2. Where the time goes

Each step loads its model **once** (`load_s`), then makes one asset per kid (`latency_s`). We use the median per kid: the first asset of each model is slower (warm-up), and the median keeps that out of the per-card time.

The card and mix steps run on the CPU, but inside the same GPU VM, so they bill at the GPU price too.

In [ ]:
CPU_STEPS = {"text", "card", "mix"}

# One card = the sum of its step times
per_card = [sum(step["latency_s"][kid] for step in steps.values()) for kid in kids]
t_gen = statistics.median(per_card)
t_load = sum(step["load_s"] or 0 for step in steps.values())
t_fixed = T_START_S + t_load

print(f"{'step':<13}{'runs on':<9}{'load s':>9}{'first s':>9}{'median s':>10}{'% of card':>11}")
for name, step in steps.items():
    load = f"{step['load_s']:.1f}" if step["load_s"] is not None else "-"
    first = step["latency_s"][kids[0]]
    share = 100 * step["median_s"] / t_gen
    where = "CPU" if name in CPU_STEPS else "GPU"
    print(f"{name:<13}{where:<9}{load:>9}{first:>9.1f}{step['median_s']:>10.1f}{share:>10.1f}%")

print(f"\nfixed, once per run: startup {T_START_S:.1f} s + model loads {t_load:.1f} s "
      f"= {t_fixed:.1f} s")
print(f"marginal, once per card: median {t_gen:.1f} s")

How much of the billed time is the fixed part, for a run of N cards? No price needed yet: these are GPU-seconds.

In [ ]:
for n in (1, 10, 20, 200):
    billed = t_fixed + n * t_gen
    print(f"N={n:<4} billed ≈ {billed / 60:6.1f} min, fixed part {100 * t_fixed / billed:5.1f}%")

## 3. The analysis at regular prices

- Marginal cost per card: `c = rate × t_gen`.
- Fixed cost per run: `F = rate × (t_start + t_load)`. An Endpoint left idle adds `rate × t_idle`.
- Total per card for N cards: `F / N + c`.

**Assumptions:** one startup per run, cards made one after another, no idle time.
**Left out:** failed or retried runs, engineering and ops time, storage and egress, and quality differences against an API. Each one makes self-hosting look cheaper than it is.

`c` is the *marginal* cost, never "the cost per card": with N = 1, the fixed part is most of the bill.

### 3.1 Cost per card

In [ ]:
from common import calculate_cost

COUNTS = (1, 10, 20, 200)
rate = PRICES[MAIN_GPU]["regular"]
cost = calculate_cost(T_START_S, t_load, per_card, rate, counts=COUNTS)
print(f"{MAIN_GPU}, regular price {rate:.2f} USD/h ({PRICE_DATE}): "
      f"c = ${cost['marginal_compute_usd']:.4f} (marginal), F = ${cost['fixed_run_usd']:.2f}")
for n, usd in cost["projected_compute_usd_per_asset"].items():
    print(f"  N={n:<4} ${usd:.4f} per card, ${usd * int(n):.2f} for the run")

santa = 200_000
print(f"\n{santa:,} cards on one {MAIN_GPU}: {santa * t_gen / 3600:,.0f} GPU-hours, "
      f"${(t_fixed + santa * t_gen) * rate / 3600:,.0f}")

### 3.2 What to optimize

Each change below moves one term of `F / N + c`. The scenarios are best cases, not measurements: Module 3 measures them.

- **Weights already on disk** (cached or baked in): removes most of the clip step's load time from `F`. Best case: clip load = 0.
- **A faster clip** (fewer steps or frames, a faster kernel, another GPU): shrinks `c`. Here: twice as fast.
- **K parallel workers**: every worker pays `F` again. You buy wall time with money.

In [ ]:
clip = steps["clip"]
scenarios = {
    "as measured": (t_load, t_gen),
    "clip weights already on disk": (t_load - clip["load_s"], t_gen),
    "clip twice as fast": (t_load, t_gen - clip["median_s"] / 2),
}
print(f"{MAIN_GPU}, regular price")
for name, (load, gen) in scenarios.items():
    per_10 = calculate_cost(T_START_S, load, [gen], rate, counts=(10,))
    print(f"{name:<30} N=10: ${per_10['projected_compute_usd_per_asset']['10']:.3f} per card")

N = 200
print(f"\n{N} cards split across K workers, each paying the fixed part:")
for k in (1, 4, 10):
    wall_h = (t_fixed + N * t_gen / k) / 3600
    usd = rate / 3600 * (k * t_fixed + N * t_gen)
    print(f"K={k:<3} wall time {wall_h:4.1f} h, cost ${usd:.2f}")

### 3.3 Self-hosted vs a per-call API: the clip step

The clip is most of the card's cost, so we compare that step, and only that step: the API sells the same model, Wan 2.2 A14B image-to-video. A one-step run has the same shape as your homework: one model, one load, N assets.

The API prices per **video second** by resolution, counted at 16 frames per second. Our clip is 480p and 81 frames. The price is the one in cell 0, checked on the date there; the provider can change it at any time.

Breakeven N = `ceil(F / (p − c))`: the number of clips per run above which the self-hosted run is cheaper. `0` means the API costs less than your marginal cost at any N.

In [ ]:
from models import WAN_I2V

video_s = WAN_I2V.num_frames / API["frames_per_video_s"]
p = API["usd_per_video_s"] * video_s
print(f"API: {API['provider']} {API['model']}, {API['resolution']}, "
      f"${API['usd_per_video_s']} per video s x {video_s:.2f} s = ${p:.4f} per clip "
      f"(checked {API['checked']}, {API['link']})")

clip_times = list(clip["latency_s"].values())
clip_cost = calculate_cost(
    T_START_S, clip["load_s"], clip_times, rate, counts=(1, 10, 20, 200), api_price=p,
)
print(f"self-hosted clip on {MAIN_GPU}, regular price: "
      f"c = ${clip_cost['marginal_compute_usd']:.4f}, F = ${clip_cost['fixed_run_usd']:.2f}")
for n, usd in clip_cost["projected_compute_usd_per_asset"].items():
    print(f"  N={n:<4} ${usd:.4f} per clip vs API ${p:.4f}")
print(f"breakeven: {clip_cost['breakeven_assets']} clips per run")

### 3.4 Same cards, different GPUs

The same worker image and the same 20 kids, run once on each GPU. Cost per asset is price × time: a slower GPU can make a cheaper card. A run without the clip step (it doesn't fit the GPU) gets no card cost.

`gpu_costs()` returns the numbers for one GPU at one price; section 4 reuses it for spot prices.

In [ ]:
ALL_STEPS = {"text", "illustration", "card", "clip", "greeting", "mix"}


def gpu_costs(gpu: str, pricing: str) -> dict | None:
    """Card and clip costs for one GPU's run at 'regular' or 'spot' prices; None if no clip."""
    s = load_run(gpu)["steps"]
    if set(s) != ALL_STEPS:
        return None
    usd_h = PRICES[gpu][pricing]
    names = list(s["text"]["latency_s"])
    cards = [sum(step["latency_s"][kid] for step in s.values()) for kid in names]
    load = sum(step["load_s"] or 0 for step in s.values())
    card = calculate_cost(T_START_S, load, cards, usd_h, counts=(20, 200))
    clip_only = calculate_cost(
        T_START_S, s["clip"]["load_s"], list(s["clip"]["latency_s"].values()), usd_h,
        api_price=p,
    )
    return {
        "usd_h": usd_h,
        "clip_load_s": s["clip"]["load_s"],
        "clip_s": s["clip"]["median_s"],
        "card_s": statistics.median(cards),
        "c": card["marginal_compute_usd"],
        "n20": card["projected_compute_usd_per_asset"]["20"],
        "n200": card["projected_compute_usd_per_asset"]["200"],
        "santa": (T_START_S + load + 200_000 * statistics.median(cards)) * usd_h / 3600,
        "breakeven": clip_only["breakeven_assets"],
    }


print(f"regular prices, {PRICE_DATE}")
print(f"{'GPU':<14}{'USD/h':>7}{'clip load s':>13}{'clip s':>8}{'card s':>8}"
      f"{'c':>9}{'N=20':>9}{'N=200':>9}{'breakeven':>11}")
for gpu in PRICES:
    r = gpu_costs(gpu, "regular")
    if r is None:
        sana = load_run(gpu)["steps"]["illustration"]["median_s"]
        print(f"{gpu:<14}no clip step: illustration {sana:.2f} s")
        continue
    print(f"{gpu:<14}{r['usd_h']:>7.2f}{r['clip_load_s']:>13.0f}{r['clip_s']:>8.1f}"
          f"{r['card_s']:>8.1f}{r['c']:>9.4f}{r['n20']:>9.4f}{r['n200']:>9.4f}{r['breakeven']:>11}")

print("\nIllustration step (Sana), cost per image at the regular price:")
for gpu, price in PRICES.items():
    sana = load_run(gpu)["steps"]["illustration"]
    print(f"{gpu:<14}load {sana['load_s']:5.1f} s, {sana['median_s']:5.2f} s per image, "
          f"${price['regular'] * sana['median_s'] / 3600:.5f}")

## 4. What spot prices change

Preemptible GPUs bill at the spot price. It's usually below the regular price, but how far below differs by GPU and region, it moves during the day, and it stays at least one cent below the regular price. Nebius can also take a preemptible GPU back: the Job ends in `ERROR`, and a rerun pays the fixed part again.

The same numbers as section 3, at the spot prices in cell 0:

In [ ]:
print(f"spot snapshot {SPOT_SNAPSHOT} vs regular, {PRICE_DATE}")
print(f"{'GPU':<14}{'regular':>9}{'spot':>7}{'saving':>8}{'c reg':>9}{'c spot':>9}"
      f"{'N=20 reg':>10}{'N=20 spot':>11}{'200k reg':>10}{'200k spot':>11}{'breakeven':>11}")
for gpu in PRICES:
    reg, spt = gpu_costs(gpu, "regular"), gpu_costs(gpu, "spot")
    if reg is None:
        continue
    saving = 1 - spt["usd_h"] / reg["usd_h"]
    print(f"{gpu:<14}{reg['usd_h']:>9.2f}{spt['usd_h']:>7.2f}{saving:>8.0%}"
          f"{reg['c']:>9.4f}{spt['c']:>9.4f}{reg['n20']:>10.4f}{spt['n20']:>11.4f}"
          f"{reg['santa']:>10,.0f}{spt['santa']:>11,.0f}"
          f"{reg['breakeven']:>6} -> {spt['breakeven']}")

cheapest = {
    pricing: min((g for g in PRICES if gpu_costs(g, pricing)),
                 key=lambda g: gpu_costs(g, pricing)["c"])
    for pricing in ("regular", "spot")
}
print(f"\nlowest marginal cost per card: {cheapest['regular']} at regular prices, "
      f"{cheapest['spot']} at spot prices")

## 5. Optional practice: your own run

One model, 10 assets in one run: one startup, one model load, ten generations. Run it any way from this module: an Endpoint called from your laptop, a Job, or a DevLab. The calculation is the same for all three; only where you read the times differs:

| You ran | `t_start` | `t_ready` | `t_gen` per asset |
| --- | --- | --- | --- |
| Job | `created_at` → `started_at` in `nebius ai job get <job_ID> --format json` | `load_s` in your `metrics_batch.json`, or your own load timer | `latency_s`, or your own timer around each generation |
| Endpoint | **Create** → `RUNNING`, from the console | The readiness wait: `RUNNING` → the first answer from `/v1/models` | The time per request, measured from your laptop |
| DevLab | **Create** → `RUNNING`, from the console | Your timer around the model load | Your timer around each generation |

Record how you ran it, the model, the GPU and the region with the times. Use the price your run actually paid: the regular price, or the spot price at the time of the run if it was preemptible.

In [ ]:
MY_RUN = {
    "model": "",  # e.g. "Sana"
    "ran_as": "",  # "Endpoint", "Job" or "DevLab"
    "gpu": "",  # e.g. "L40S"
    "region": "",  # e.g. "eu-north1"
    "pricing": "",  # "regular" or "spot"
}
MY_T_START = None  # s, see the table above
MY_T_READY = None  # s, see the table above
MY_T_GEN = []  # s per asset: the 10 generations of your run
MY_RATE = None  # USD per hour your run paid
MY_API_PRICE = None  # USD per asset for the same or closest model, from its pricing page
MY_API_LINK = ""
assert all(MY_RUN.values()), "Record model, ran_as, gpu, region and pricing"
assert None not in (MY_T_START, MY_T_READY, MY_RATE, MY_API_PRICE), "Fill in your numbers"
assert len(MY_T_GEN) == 10 and MY_API_LINK, "Add your 10 generation times and the API link"

mine = calculate_cost(
    MY_T_START, MY_T_READY, MY_T_GEN, MY_RATE, counts=(10,), api_price=MY_API_PRICE
)
print(", ".join(f"{k}: {v}" for k, v in MY_RUN.items()))
print(f"generation time per asset (median): {statistics.median(MY_T_GEN):.2f} s")
print(f"marginal cost per asset c: ${mine['marginal_compute_usd']:.5f}")
print(f"fixed cost per run F: ${mine['fixed_run_usd']:.4f}")
print(f"cost per asset at N=10: ${mine['projected_compute_usd_per_asset']['10']:.4f}")
print(f"API: ${MY_API_PRICE} per asset, {MY_API_LINK}")
print(f"breakeven N: {mine['breakeven_assets']}")

## 6. Check what you spent

The console shows your balance under **Billing** and the spend per project, service and resource under **Billing → Usage**. Compare the usage for your run with `rate × billed seconds` from section 2. The usage view is hourly, so it won't match to the second.

You're done when:

- [ ] You have the per-step table for one run.
- [ ] You have the cost per card at N = 1, 10, 20 and 200 at regular prices, extrapolated from that run.
- [ ] You have the cost per card on each GPU and the clip breakeven against the API, from our example runs.
- [ ] You know what spot prices change.
- [ ] Optional: you have `c`, the cost per asset at N = 10 and the breakeven for your own run.